# Single Bias-Experiment Runner

Runs one bias-evaluation experiment end-to-end. Accepts a YAML config file specifying the bias type (length / position / sycophancy / uncertainty), reward model, dataset source, and output directories, as argument to its `run(...)` function.

Should run on any `transformers`-supported GPU (Nvidia, Apple Silicon) or CPU (we discourage use of CPUs due to very high inference latency) if you use the multi-GPU branch.

This particular example required ~4 GB on a T4 GPU and took less than 30 minutes to run end-to-end.

**Summary:**

- Loads the corresponding `BiasExperiment` subclass, trains a linear probe on a held-out probe set, evaluates on the test split, saves the probe artifact, and writes diagnostic plots.
- Returns an `ExperimentResults` object containing summary statistics of the run.
- Handles cross-dataset generalisation by pointing the probe trainer at one dataset and the evaluator at another.

In [ ]:
# Clone ideally the `multigpu` branch
!git clone -b feature/multigpu https://github.com/nondatur/OneBiasAfterAnotherFork.git

Google Colab uses `/content` as its home directory. If you run this on a different Juypter fork, you might need to modify:

In [ ]:
# We specify these two global variables:
HF_HOME   = "/content/huggingface"
REPO_HOME = "/content/OneBiasAfterAnotherFork"

In [ ]:
import sys
import os
from pathlib import Path

# Set up HF home (for Colab only)
os.environ['HF_HOME'] = HF_HOME
if HF_HOME not in sys.path:
    sys.path.append(HF_HOME)

# Ensure the repository root is in the system path
repo_path = os.path.abspath(REPO_HOME)
if repo_path not in sys.path:
     sys.path.append(repo_path)

# Set up the project root
PROJECT_ROOT = Path(REPO_HOME).resolve()
if str(PROJECT_ROOT) not in sys.path:
     sys.path.insert(0, str(PROJECT_ROOT))

# Check system path(s)
for path in sys.path:
    print(f"Current sys.path(s): {path}")

Current sys.path: ['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/huggingface', '/content/OneBiasAfterAnotherFork']


In [ ]:
from __future__ import annotations

import logging
import sys
import os
from pathlib import Path
from typing import Type
import yaml

try:
    from src.nb.experiments.base import BiasExperiment, ExperimentConfig, ExperimentResults  # type: ignore
    from src.nb.experiments.length import LengthBiasExperiment  # type: ignore
    from src.nb.experiments.sycophancy import SycophancyBiasExperiment  # type: ignore
    from src.nb.experiments.uncertainty import UncertaintyBiasExperiment  # type: ignore
    from src.nb.experiments.position import (  # type: ignore
        PositionBiasExperiment,
        BinaryPositionBiasExperiment,
        FreeformPositionBiasExperiment,
    )
    print("Successfully imported src.nb modules.")
except ImportError as e:
    print(f"Import failed: {e}")
    print(f"Current sys.path: {sys.path}")
    print(f"Contents of {PROJECT_ROOT}: {os.listdir(PROJECT_ROOT) if PROJECT_ROOT.exists() else 'Directory not found'}")
    raise e

EXPERIMENT_CLASSES: dict[str, Type[BiasExperiment]] = {
    'length': LengthBiasExperiment,
    'sycophancy': SycophancyBiasExperiment,
    'uncertainty': UncertaintyBiasExperiment,
    'position': PositionBiasExperiment,
}

Successfully imported src.nb modules.


In [ ]:
def _load_defaults(section: str) -> dict:
    """Return default config values for *section* from configs/default_values.yaml."""
    with open(PROJECT_ROOT / "configs" / "default_values.yaml") as f:
        import yaml as _yaml
        return _yaml.safe_load(f).get(section, {})

In [ ]:
def run(config_path: "str | Path | None" = None, **overrides):
    """Run a bias evaluation experiment.

    Args:
        config_path: Path to an experiment YAML config.  If relative, resolved
                     from the project-root ``configs/`` directory.
        **overrides: Per-run keyword overrides, e.g. ``device="cpu"``.

    Returns:
        ExperimentResults
    """
    cfg = _load_defaults("run_experiment")

    if config_path is not None:
        p = Path(config_path)
        if not p.is_absolute():
            p = PROJECT_ROOT / "configs" / p
        with open(p) as f:
            cfg.update(yaml.safe_load(f))

    cfg.update(overrides)

    config = ExperimentConfig.from_dict(cfg)

    # Print experiment config
    print("-----------------------------------")
    print("Experiment :", config.name)
    print("Bias type  :", config.bias_type)
    print("Model      :", config.model_path)
    print("Dataset    :", config.dataset_source)
    print("-----------------------------------")

    exp_cls = EXPERIMENT_CLASSES.get(config.bias_type)
    if exp_cls is None:
        raise ValueError(
            f"Unknown bias type: {config.bias_type!r}. "
            f"Choose from: {list(EXPERIMENT_CLASSES)}"
        )

    if config.bias_type == "position":
        if config.dataset_class in (
            "position_freeform",
            "position_freeform_bigbench",
            "position_freeform_plausibleqa",
        ):
            exp_cls = FreeformPositionBiasExperiment
            print("Using freeform position bias experiment")
        elif config.dataset_class == "position_plausibleqa" or (
            "plausibleqa" in (config.dataset_source or "").lower()
            and not config.dataset_class
        ):
            exp_cls = BinaryPositionBiasExperiment
            print("Using binary position bias experiment for PlausibleQA")

    if cfg.get("probe_source"):
        print(
            "Cross-dataset: probe from %s, eval on %s",
            cfg["probe_source"], config.dataset_source,
        )
        config.extra["probe_source"] = cfg["probe_source"]
        if cfg.get("probe_extra"):
            config.extra["probe_extra"] = cfg["probe_extra"]

    experiment = exp_cls(config)
    return experiment.run()

In [ ]:
experiment = run("position_deberta_gsm8k.yaml") # reads them from `configs/position_deberta_gsm8k.yaml`

-----------------------------------
Experiment : position_bert_2_gsm8k
Bias type  : position
Model      : OpenAssistant/reward-model-deberta-v3-large-v2
Dataset    : guipenedo/gsm8k-mc
-----------------------------------


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: OpenAssistant/reward-model-deberta-v3-large-v2
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Computing rewards (baseline+nulled): 100%|██████████| 1319/1319 [02:46<00:00,  7.90it/s]


In [ ]:
import json
print(json.dumps(experiment.to_dict(), indent=4))

{
    "config": {
        "name": "position_bert_2_gsm8k",
        "bias_type": "position",
        "model_path": "OpenAssistant/reward-model-deberta-v3-large-v2",
        "trust_remote_code": true,
        "dataset_source": "guipenedo/gsm8k-mc",
        "dataset_class": "",
        "probe_size": 500,
        "max_test_examples": null,
        "split_seed": 42,
        "null_alpha": 1.0,
        "batch_size": 4,
        "max_length": 2048,
        "device": "cuda",
        "raw_data_dir": "artifacts/raw_data",
        "artifacts_dir": "artifacts",
        "plots_dir": "plots",
        "save_probe": true,
        "extra": {
            "dataset_id": "guipenedo/gsm8k-mc",
            "train_split": "train",
            "eval_split": "test"
        }
    },
    "baseline": {
        "accuracy": 0.24184988627748294,
        "position_distribution": [
            1.6679302501895377,
            8.642911296436695,
            20.84912812736922,
            68.84003032600455
        ],
      